<a href="https://colab.research.google.com/github/ryanglenc/MAT_442/blob/main/3_4_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

## 3.4.1 The Logistic Model

Linear regression predicts a number, but for a binary label y ∈ {0, 1} we want a **probability**. Logistic regression passes a linear score through the **sigmoid** function:

σ(z) = 1 / (1 + e^(−z)),  P(y = 1 | x) = σ(wᵀx)

The sigmoid maps any real number into (0, 1), is increasing, and satisfies σ(0) = 0.5. So the prediction "class 1" corresponds to wᵀx > 0, which means **the decision boundary is the line (hyperplane) wᵀx = 0**.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.linspace(-8, 8, 400)
plt.plot(z, sigmoid(z), color='steelblue')
plt.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('z = w·x')
plt.ylabel('sigma(z)')
plt.title('The sigmoid function')
plt.show()

print("sigmoid(0) =", sigmoid(0))
print("sigmoid(-5), sigmoid(5) =", sigmoid(-5), sigmoid(5))

## Example Data

We generate two overlapping clusters of 2D points (class 0 and class 1). Because the classes overlap, no line separates them perfectly, which is the realistic case. We add a column of ones to the data matrix so the intercept w₀ is part of the weight vector.

In [ ]:
n_per_class = 100
X0 = rng.normal(loc=[-1.0, -1.0], scale=1.0, size=(n_per_class, 2))
X1 = rng.normal(loc=[1.5, 1.0], scale=1.0, size=(n_per_class, 2))

X = np.vstack((X0, X1))
y = np.concatenate((np.zeros(n_per_class), np.ones(n_per_class)))

# Design matrix with intercept column
Xb = np.column_stack((np.ones(len(X)), X))
m = len(y)

plt.scatter(X0[:, 0], X0[:, 1], color='tab:blue', alpha=0.7, label='Class 0')
plt.scatter(X1[:, 0], X1[:, 1], color='tab:orange', alpha=0.7, label='Class 1')
plt.xlabel('x1'); plt.ylabel('x2')
plt.title('Two-class data')
plt.legend()
plt.show()

## 3.4.2 Maximum Likelihood and the Loss Function

Each label is Bernoulli with success probability pᵢ = σ(wᵀxᵢ). The likelihood of the data is

L(w) = ∏ pᵢ^yᵢ (1 − pᵢ)^(1−yᵢ)

Maximizing the log-likelihood is the same as minimizing the **average negative log-likelihood (cross-entropy loss)**:

J(w) = −(1/m) Σ [ yᵢ log pᵢ + (1 − yᵢ) log(1 − pᵢ) ]

Its gradient has a clean form:

∇J(w) = (1/m) Xᵀ (p − y)

and its Hessian is

∇²J(w) = (1/m) Xᵀ S X,  where S = diag(pᵢ(1 − pᵢ)).

Since every pᵢ(1 − pᵢ) > 0, the Hessian is positive semidefinite, so **J is convex**. By Section 3.3.2, any minimizer we find is a global minimizer.

Unlike linear regression, setting ∇J = 0 has no closed-form solution, so we solve it iteratively.

In [ ]:
def loss(w):
    """Average negative log-likelihood, written in a numerically stable form:
    -[y log p + (1-y) log(1-p)] = log(1 + e^z) - y z"""
    z = Xb @ w
    return np.mean(np.logaddexp(0, z) - y * z)

def gradient(w):
    p = sigmoid(Xb @ w)
    return Xb.T @ (p - y) / m

def hessian(w):
    p = sigmoid(Xb @ w)
    S = p * (1 - p)
    return (Xb.T * S) @ Xb / m

# Sanity check: compare the analytic gradient to a finite-difference estimate
w_test = rng.normal(size=3)
eps = 1e-6
numeric_grad = np.array([(loss(w_test + eps * e) - loss(w_test - eps * e)) / (2 * eps)
                         for e in np.eye(3)])
print("Analytic gradient:  ", gradient(w_test))
print("Numerical gradient: ", numeric_grad)
print("Match:", np.allclose(gradient(w_test), numeric_grad, atol=1e-6))

# Convexity check: Hessian eigenvalues at a random point should all be > 0
print("\nHessian eigenvalues at a random w:", np.linalg.eigvalsh(hessian(w_test)))

## 3.4.3 Fitting by Gradient Descent

Starting from w = 0, we repeat

w ← w − α ∇J(w)

This is exactly the algorithm from Section 3.3.3, now applied to the logistic loss. Because J is convex, gradient descent with a small enough step size converges to the global minimizer.

In [ ]:
def gradient_descent(w0, lr, n_iter):
    w = np.array(w0, dtype=float)
    losses = [loss(w)]
    for _ in range(n_iter):
        w = w - lr * gradient(w)
        losses.append(loss(w))
    return w, np.array(losses)

w_gd, losses_gd = gradient_descent(np.zeros(3), lr=0.5, n_iter=5000)

print("Weights from gradient descent [w0, w1, w2]:", w_gd)
print("Final loss:", losses_gd[-1])
print("Gradient norm at solution (should be near 0):", np.linalg.norm(gradient(w_gd)))

plt.plot(losses_gd, color='darkorange')
plt.xlabel('Iteration')
plt.ylabel('Cross-entropy loss J(w)')
plt.title('Gradient descent on the logistic loss')
plt.show()

## 3.4.4 Fitting by Newton's Method

Because we have the Hessian, we can use a faster second-order method:

w ← w − [∇²J(w)]⁻¹ ∇J(w)

Newton's method typically converges in only a handful of iterations on this problem (in the statistics literature this is called iteratively reweighted least squares). We then compare both solutions with scikit-learn, with regularization turned off so it solves the same problem.

In [ ]:
w_newton = np.zeros(3)
for i in range(10):
    g = gradient(w_newton)
    H = hessian(w_newton)
    w_newton = w_newton - np.linalg.solve(H, g)
    print(f"Iteration {i+1}: loss = {loss(w_newton):.8f}, gradient norm = {np.linalg.norm(g):.2e}")

print("\nWeights from Newton's method:", w_newton)

# Compare with scikit-learn (no regularization)
sk = LogisticRegression(penalty=None, max_iter=1000).fit(X, y)
w_sk = np.concatenate((sk.intercept_, sk.coef_.ravel()))

print("\nGradient descent:", w_gd)
print("Newton's method: ", w_newton)
print("scikit-learn:    ", w_sk)
print("\nMax difference, Newton vs scikit-learn:", np.max(np.abs(w_newton - w_sk)))
print("Max difference, gradient descent vs Newton:", np.max(np.abs(w_gd - w_newton)))

## 3.4.5 Decision Boundary and Evaluation

With fitted weights w, we predict class 1 when P(y = 1 | x) > 0.5, which is the same as w₀ + w₁x₁ + w₂x₂ > 0. The boundary is the line

x₂ = −(w₀ + w₁x₁) / w₂.

We plot it over the data and measure performance with **accuracy** and a **confusion matrix**.

In [ ]:
w = w_newton   # use the converged solution

# Plot the probability surface and decision boundary
xx, yy = np.meshgrid(np.linspace(-4, 5, 300), np.linspace(-4, 4, 300))
grid = np.column_stack((np.ones(xx.size), xx.ravel(), yy.ravel()))
probs = sigmoid(grid @ w).reshape(xx.shape)

plt.contourf(xx, yy, probs, levels=20, cmap='coolwarm', alpha=0.4)
plt.colorbar(label='P(y = 1 | x)')
plt.scatter(X0[:, 0], X0[:, 1], color='tab:blue', edgecolor='k', alpha=0.8, label='Class 0')
plt.scatter(X1[:, 0], X1[:, 1], color='tab:orange', edgecolor='k', alpha=0.8, label='Class 1')

x1_line = np.linspace(-4, 5, 100)
plt.plot(x1_line, -(w[0] + w[1] * x1_line) / w[2], 'k--', linewidth=2, label='Decision boundary')
plt.xlim(-4, 5); plt.ylim(-4, 4)
plt.xlabel('x1'); plt.ylabel('x2')
plt.title('Logistic regression decision boundary')
plt.legend()
plt.show()

# Predictions and metrics
probs_train = sigmoid(Xb @ w)
y_pred = (probs_train > 0.5).astype(int)

accuracy = np.mean(y_pred == y)
TP = np.sum((y_pred == 1) & (y == 1))
TN = np.sum((y_pred == 0) & (y == 0))
FP = np.sum((y_pred == 1) & (y == 0))
FN = np.sum((y_pred == 0) & (y == 1))

print(f"Accuracy: {accuracy:.3f}")
print("\nConfusion matrix (rows = true class, columns = predicted class):")
print(np.array([[TN, FP],
                [FN, TP]]))

# Predict a new point
new_point = np.array([1.0, 0.5, 0.5])   # [intercept, x1, x2]
print(f"\nP(class 1) for the point (0.5, 0.5): {sigmoid(new_point @ w):.3f}")

## 3.4.6 Interpreting the Weights

In logistic regression, the **log-odds** of class 1 is linear in the features:

log( p / (1 − p) ) = w₀ + w₁x₁ + w₂x₂

So increasing x₁ by one unit multiplies the odds of class 1 by e^(w₁), holding x₂ fixed.

In [ ]:
print("Odds ratio for a one-unit increase in x1:", np.exp(w[1]))
print("Odds ratio for a one-unit increase in x2:", np.exp(w[2]))

# Verify: logit(p) equals the linear score
p_check = sigmoid(Xb @ w)
logit = np.log(p_check / (1 - p_check))
print("\nlogit(p) equals linear score w·x:", np.allclose(logit, Xb @ w))